In [1]:
import sys
sys.path.append("..")  # so we can import auto_pricer package

from auto_pricer.car import Car

train, val, test = Car.from_hub("ShahidHKhan/cars_lite_processed")

print(f"train: {len(train)}, val: {len(val)}, test: {len(test)}")
print(f"missing summaries -> train: {sum(c.summary is None for c in train)}, "
      f"val: {sum(c.summary is None for c in val)}, "
      f"test: {sum(c.summary is None for c in test)}")

train: 26132, val: 1452, test: 1452
missing summaries -> train: 0, val: 0, test: 0


In [3]:
import os
from dotenv import load_dotenv
from transformers import AutoTokenizer

load_dotenv(override=True)

BASE_MODEL = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=os.environ.get("HF_TOKEN"))

print(tokenizer)

TokenizersBackend(name_or_path='meta-llama/Llama-3.2-3B', vocab_size=128000, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>'}, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128005: AddedToken("<|reserved_special_token_2|>", rstrip=False, lstrip=False, single_word=False, normalized=Fa

In [4]:
for car in train + val + test:
    car.make_prompts(tokenizer, max_tokens=120, do_round=True)

print("Done.")
print(f"Sample prompt:\n{train[0].prompt}")
print(f"\nSample completion: {train[0].completion}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Done.
Sample prompt:
What does this used car cost to the nearest dollar?

Title: 2008 Porsche 911 Turbo Convertible
Category: Convertible
Make: Porsche
Description: Exceptionally well-maintained 2008 Porsche 911 Turbo convertible with a unique cocoa interior and premium features.
Details: 19,000 miles, gasoline, automatic transmission.

Price is $

Sample completion: 77695.00


In [5]:
import numpy as np

token_lengths = [len(tokenizer.encode(c.summary, add_special_tokens=False)) for c in train]
token_lengths = np.array(token_lengths)

print(f"min: {token_lengths.min()}, max: {token_lengths.max()}, mean: {token_lengths.mean():.1f}")
print(f"rows exceeding 120 tokens (will be truncated): {(token_lengths > 120).sum()} / {len(token_lengths)}")

min: 41, max: 146, mean: 58.5
rows exceeding 120 tokens (will be truncated): 1 / 26132


In [6]:
from auto_pricer.car import Car

HF_USER = "ShahidHKhan"
Car.push_prompts_to_hub(f"{HF_USER}/cars_lite_prompts", train, val, test)
print("Pushed.")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed.


In [7]:
from datasets import load_dataset

ds = load_dataset(f"{HF_USER}/cars_lite_prompts")
print(ds)

print("\nSample row:")
print(ds["train"][0])

README.md:   0%|          | 0.00/509 [00:00<?, ?B/s]

d:\mrsha\Projects\AutoPricer\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mrsha\.cache\huggingface\hub\datasets--ShahidHKhan--cars_lite_prompts. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/122k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/123k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26132 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1452 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1452 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 26132
    })
    val: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1452
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1452
    })
})

Sample row:
{'prompt': 'What does this used car cost to the nearest dollar?\n\nTitle: 2008 Porsche 911 Turbo Convertible\nCategory: Convertible\nMake: Porsche\nDescription: Exceptionally well-maintained 2008 Porsche 911 Turbo convertible with a unique cocoa interior and premium features.\nDetails: 19,000 miles, gasoline, automatic transmission.\n\nPrice is $', 'completion': '77695.00'}
